## NLST CT Abnormalities → Clean Nodule-Level Dataset (Set-1 / Set-2)

### Overview

This script preprocesses NLST participant and spiral CT abnormality tables to construct a cleaned **nodule-level analysis dataset** focused on **non-calcified nodules ≥ 4 mm** (abnormality code `51`). It merges participant metadata with CT abnormality records, decodes categorical fields into human-readable labels, applies inclusion/exclusion criteria, engineers clinically relevant features, splits the cohort by NLST CT set (Set-1 vs Set-2), and exports two final CSV files for downstream ML/statistical analysis.

---

### Inputs

The script expects the following CSV files under `DF_SCR_PATH = 'scr_dir/'`:

1. **Participants table**

   * `participant_d040722.csv`
   * Used to extract demographics, screening/cancer variables, family history, emphysema, and primary cancer locations.

2. **Spiral CT abnormalities table**

   * `Spiral CT Abnormalities/sct_abnormalities_d040722.csv`
   * Used to extract abnormality description, nodule size, type, margin, and location.

---

### Key Processing Steps

#### 1) Decode and filter participant cohort

* Decodes:

  * `gender` using `GENDER_DECODER`
  * `race` using `RACE_DECODER`
  * screening group and cancer screening outcome using `screening_group_decoder` and `can_scr_decoder`
* Filters to CT study arm only (`rndgroup == 1`).

#### 2) Derive primary cancer locations

* Converts multiple primary cancer location indicator columns (e.g., `locrup`, `locllow`) to Yes/No.
* Generates:

  * `primary_cancer_locations`: comma-separated list of lung locations marked “Yes”
  * time-indexed lung cancer flags: `lung_cancer_t0`, `lung_cancer_t1`, `lung_cancer_t2` (from `cancyr`)

#### 3) Merge CT abnormalities with participant metadata

* Merges abnormalities with participants on `pid`.
* Decodes nodule descriptors:

  * `sct_epi_loc` (lobe), `sct_pre_att` (attenuation/type), `sct_margins` (margin)

#### 4) Match abnormality location with primary cancer location (optional feature)

* Maps lobe strings to location form fields and computes:

  * `match_primary`: 1 if abnormality lobe matches any primary cancer location, else 0.

#### 5) Inclusion/exclusion criteria and cleaning

The script prints cohort counts at each step and applies the following filters:

* Keep only abnormality description `51` (non-calcified nodule/mass ≥ 4 mm).
* Drop missing values in:

  * `sct_long_dia` (nodule size)
  * `sct_pre_att` (nodule type)
  * `all_sct_set` (CT Set assignment)
  * `Family_History` (derived)
  * `diagemph` (emphysema indicator)
* Keep only clinically relevant nodule types:

  * `solid`, `part-solid`, `ground-glass` (drops “others/unknown”)

---

### Feature Engineering (Final Columns)

The final dataset is built from:

`['pid','study_yr','age','gender','all_sct_set','sct_long_dia','sct_margins','sct_pre_att','sct_epi_loc','can_scr','diagemph','Family_History']`

And adds:

* `Family_History`: max across (`fambrother`, `famchild`, `famfather`, `fammother`, `famsister`)
* `Nodule_Type`: mapped from `sct_pre_att` → {solid, part-solid, ground-glass}
* `Spiculation`: binary from `sct_margins` (1 = spiculated, else 0)
* `Upper_Lobe`: binary mapped from lobe location (1 = upper/middle, else 0)
* `Cancer_lbl`: binary label derived from `can_scr` (0 = “No Cancer”, 1 = otherwise)

---

### Outputs

Two CSV files are exported to `save_dir = 'output_dir/'`:

* `nlst_ct_nodule_df_set1.csv`
* `nlst_ct_nodule_df_set2.csv`

These correspond to NLST CT Set assignment (`all_sct_set == 1` and `all_sct_set == 2`), and the script prints summary counts (unique patients, abnormality rows, and cancer-positive rows) for each set.

---

### Configuration

Filtering toggles exist but are not actively used in the current script logic:

* `FILTER_BY_YEAR`
* `FILTER_BY_GENDER`
* `FILTER_BY_ABNORMALITIES`
* `PLOT`

---

### Notes / Assumptions

* Abnormality code `51` is treated as the primary inclusion criterion (≥ 4 mm non-calcified nodules).
* Cancer label is derived from screening outcome text after decoding (`can_scr`).
* The script assumes file paths and column names match NLST-formatted exports.




# CODE

In [ ]:
###################################################--Libaries---#######################################################

import os
import glob
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from tabulate import tabulate
import matplotlib.pyplot as plt
from tabulate import tabulate
#plt.rcParams.update({'font.size': 8})



## Functions ###

## 统计在某一列中，各个取值出现了几次
def percentage_and_count(df,colmn):
        ###############--Race-#######
    # Perform value counts on the 'race_decoce' column and normalize the counts
    total_counts            = df[colmn].value_counts(normalize=False)
    total_percentage        = df[colmn].value_counts(normalize=True)

    
    count_number         = total_counts.values
    count_percentage     = total_percentage.values
    count_keys           = total_counts.keys()

    # Extract the keys (unique values) from the value counts result
    # Create a list of lists with keys and counts
    data = [[key, count,count2] for key, count,count2 in zip(count_keys,count_number,count_percentage)]
    headers = [colmn,"#", "(%)"]
    print(tabulate(data,headers, tablefmt="github"))
    return

  


################################################## --Directories---#######################################################
DF_SCR_PATH       ='scr_dir/'




#####################################################--Filtering CONDITIONS---#######################################################

FILTER_BY_YEAR            = False # Filter by screening year
FILTER_BY_GENDER          = False  # Filter by screening year
FILTER_BY_ABNORMALITIES   = False # Filter by screening year
PLOT                      = True


######################################################## COLMN NAMES ##################################################

PATIENT_ID_COLMN_NAME                      = 'pid'
PATIENT_GENDER_COLMN_NAME                  = 'gender'
PATIENT_AGE_COLMN_NAME                     = 'age'
PATIENT_RACE_COLMN_NAME                    = 'race'
PATIENT_STUDYYEAR_COLMN_NAME               ='study_yr'
PATIENT_ABNORNALITY_DESCRIPTION_COLMN_NAME = 'sct_ab_desc'



CANCER_GRP_COLMM_NAME                       = 'scr_group'
CANCER_SCR_COLMM_NAME                       = 'can_scr'
CANCER_RPT_LINK_COLMM_NAME                  = 'canc_rpt_link' # Is the diagnosis of lung cancer associated with a positive screen?
ANY_REPORTED_CANCER_COLMM_NAME              = 'anyscr_has_nodule'
LUNG_CANCER_COLMM_NAME                      = 'lung_cancer'                                                   
NODULE_LOC_COLMM_NAME                       = "sct_epi_loc"
NODULE_MARGIN_COLMM_NAME                    = "sct_margins"
NODULE_ATT_COLMM_NAME                       = "sct_pre_att"


ABNORNALITIES_TO_CONDIDER = [51]


#####################################################--Decoder---########################################################

GENDER_DECODER    = {1 :"male",2 :"female"}

ABNORNALITY_DESCRIPTION_DECODER = {51 : "Non-calcified nodule or mass (opacity >= 4 mm diameter)",
                      52 :"Non-calcified micronodule(s) (opacity < 4 mm diameter)",
                      53 :"Benign lung nodule(s) (benign calcification)",
                      54 :"Atelectasis, segmental or greater",
                      55 :"Pleural thickening or effusion",
                      56 :"Non-calcified hilar/mediastinal adenopathy or mass (>= 10 mm on short axis)",
                      57 :"Chest wall abnormality (bone destruction, metastasis, etc.)",
                      58 :"Consolidation",
                      59 :"Emphysema",
                      60 :"Significant cardiovascular abnormality",
                      61 :"Reticular/reticulonodular opacities, honeycombing, fibrosis, scar",
                      62 :"6 or more nodules, not suspicious for cancer (opacity >= 4 mm)",
                      63 :"Other potentially significant abnormality above the diaphragm",
                      64 :"Other potentially significant abnormality below the diaphragm",
                      65 :"Other minor abnormality noted"}


RACE_DECODER        = { 1:"White",
                        2:"Black or African-American",
                        3:"Asian",
                        4:"American Indian or Alaskan Native",
                        5:"Native Hawaiian or Other Pacific Islander",
                        6:"More than one race",
                        7:"Participant refused to answer",
                        9:"Missing data form - form is not expected to ever be completed",
                        96:"Missing - no response",
                        98:"Missing - form was submitted and the answer was left blank",
                        99:"Unknown/ decline to answer"}




MARGIN_DECODER       = {1 :"Spiculated (Stellate)",
                        2 :"Smooth",
                        3 :"Poorly defined",
                        9 :"Unable to determine"}


ATTN_DECODER        ={1:"Soft Tissue",
                     2:"Ground glass",
                     3:"Mixed",
                     4:"Fluid/water",
                     6:"Fat",
                     7:"Other",
                     9:"Unable to determine"}

LOC_DECODER        ={1:"Right Upper Lobe",
                 2:"Right Middle Lobe",
                 3:"Right Lower Lobe",
                 4:"Left Upper Lobe",
                 5:"Lingula",
                 6:"Left Lower Lobe",
                 8:"Other"}


screening_group_decoder = {
    1: "Screen-detected cancer",
    2: "Has nodule(s)",
    3: "Screened, but no nodules",
    4: "Never screened",
    5: "Other lung cancer"
}


##==== cancer screening 
can_scr_decoder = {
    0: "No Cancer",
    1: "Positive Screen",
    2: "Negative Screen",
    3: "Missed Screen",
    4: "Post Screening"
}

#--------------------------------##


##====== All participants data ========#
all_participants                             = pd.read_csv(DF_SCR_PATH+'participant_d040722.csv')
# 转换性别
all_participants[PATIENT_GENDER_COLMN_NAME]  = all_participants[PATIENT_GENDER_COLMN_NAME].map(GENDER_DECODER)
# 转换种族
all_participants[PATIENT_RACE_COLMN_NAME]    = all_participants[PATIENT_RACE_COLMN_NAME].map(RACE_DECODER)
#Study arm| CT=1

# rndgroup==1 ?
all_participants_ct              = all_participants[all_participants['rndgroup']==1].reset_index(drop=True)
all_participants_ct['scr_group'] = all_participants_ct['scr_group'].map(screening_group_decoder)
all_participants_ct['can_scr']   = all_participants_ct['can_scr'].map(can_scr_decoder)


location_decoder = {
    '.N': 'Not Applicable',
    0: 'No',
    1: 'Yes'
}

# List of primary cancer location columns
primary_cancer_Loc_columns = ['locunk', 'locoth', 'locmed', 'loccar', 'loclin',
                              'locrhil', 'locrlow', 'locrmid', 'locrmsb', 'locrup',
                              'loclhil', 'locllow', 'loclmsb', 'loclup']

# 转换所有location列的值到yes/no
for col in primary_cancer_Loc_columns:
    all_participants_ct[col] = all_participants_ct[col].map(location_decoder)

# 选出所有有癌症的位置列，合并成一个新列
def get_primary_cancer_locations(row):
    locations = [col for col in primary_cancer_Loc_columns if row[col] == 'Yes']
    return ', '.join(locations) if locations else 'No primary location identified'

# Add a new column for primary cancer locations
all_participants_ct['primary_cancer_locations'] = all_participants_ct.apply(get_primary_cancer_locations, axis=1)

# 构建三个新列：代表第一次出现癌症时的studyyear
all_participants_ct['lung_cancer_t0'] = all_participants_ct['cancyr'].apply(lambda x: 1 if x == 0 else 0)
all_participants_ct['lung_cancer_t1'] = all_participants_ct['cancyr'].apply(lambda x: 1 if x == 1 else 0)
all_participants_ct['lung_cancer_t2'] = all_participants_ct['cancyr'].apply(lambda x: 1 if x == 2 else 0)





all_participants_ct_filtered = all_participants_ct
'''
[[PATIENT_ID_COLMN_NAME,PATIENT_GENDER_COLMN_NAME,PATIENT_AGE_COLMN_NAME,PATIENT_RACE_COLMN_NAME,
                                                    CANCER_GRP_COLMM_NAME,
                                                    CANCER_SCR_COLMM_NAME,
                                                    CANCER_RPT_LINK_COLMM_NAME,
                                                    ANY_REPORTED_CANCER_COLMM_NAME,
                                                    LUNG_CANCER_COLMM_NAME,
                                                    'cancyr','lung_cancer_t0','lung_cancer_t1','lung_cancer_t2',
                                                    'locunk','locoth',  'locmed',  'loccar', 'loclin',
                                                    'locrhil','locrlow','locrmid','locrmsb','locrup',
                                                    'loclhil','locllow','loclmsb','loclup','primary_cancer_locations',
                                                    'fambrother','famchild','famfather','fammother','famsister',
                                                   'diagemph']]
'''
primary_cancer_Loc_columns = 'locunk','locoth',  'locmed',  'loccar', 'loclin','locrhil','locrlow','locrmid','locrmsb','locrup','loclhil','locllow','loclmsb','loclup'
                              



#-读取CT abnormal数据并合并
nlst_ctab_idc  = pd.read_csv(DF_SCR_PATH+"Spiral CT Abnormalities/sct_abnormalities_d040722.csv")
nlst_ctab_idc  = nlst_ctab_idc.merge(all_participants_ct_filtered, how='left',on='pid')

    
#--Mapping
# 转换nodule的位置列
nlst_ctab_idc[NODULE_LOC_COLMM_NAME]=nlst_ctab_idc[NODULE_LOC_COLMM_NAME].map(LOC_DECODER)
# 转换nodule的类型列
nlst_ctab_idc[NODULE_ATT_COLMM_NAME]= nlst_ctab_idc[NODULE_ATT_COLMM_NAME].map(ATTN_DECODER)
# 转换nodule的边缘特征列
nlst_ctab_idc[NODULE_MARGIN_COLMM_NAME]=nlst_ctab_idc[NODULE_MARGIN_COLMM_NAME].map(MARGIN_DECODER)


primaimay_loc_DECODER    ={"Right Upper Lobe":'locrup',
                 "Right Middle Lobe":'locrmid',
                 "Right Lower Lobe":'locrlow',
                 "Left Upper Lobe":'loclup',
                 "Lingula": 'loclin',
                 "Left Lower Lobe":'locllow',
                 "Other":'locoth'}    
    
# 构建新列'sct_epi_loc_sform'，将位置映射到和all_participants_ct中primary_cancer_locations列一致的格式
nlst_ctab_idc['sct_epi_loc_sform']=nlst_ctab_idc[NODULE_LOC_COLMM_NAME].map(primaimay_loc_DECODER)


# Function to check if the mapped value in COLMN1 matches any of the values in COLMN2
# 若nodule位置在cancer location中，返回1
def match_lobe_mapped(row):
    colmn1_mapped_value = row['sct_epi_loc_sform']  # Assuming this is already mapped
    colmn2_values = row['primary_cancer_locations'].split(',') if pd.notna(row['primary_cancer_locations']) else []
    if pd.isna(colmn1_mapped_value):
        return 0  # No lobe to match
    return 1 if colmn1_mapped_value in colmn2_values else 0



nlst_ctab_idc['match_primary'] = nlst_ctab_idc.apply(match_lobe_mapped, axis=1)



############--- Filtering by nodules, and years------ ####


#---- Filtered by non-calcified nodule > 4mm #### 
all_nlst_ctab_idc  = nlst_ctab_idc

##print('Inclusion-exclusion-Crieteria')

print('1) all data |patient:{}|abnornalities entried:{}'.format(len(all_nlst_ctab_idc['pid'].unique()),len(nlst_ctab_idc)))

nlst_ctab_idc      = nlst_ctab_idc[nlst_ctab_idc['sct_ab_desc'].isin([51])].reset_index(drop=True)

# 按照studyyr划分三个子集
nlst_ctab_idc_t0   = nlst_ctab_idc[nlst_ctab_idc['study_yr']==0].reset_index(drop=True)
nlst_ctab_idc_t1   = nlst_ctab_idc[nlst_ctab_idc['study_yr']==1].reset_index(drop=True)
nlst_ctab_idc_t2   = nlst_ctab_idc[nlst_ctab_idc['study_yr']==2].reset_index(drop=True)

print('2) Filtered by non-calcified nodule > 4mm |patient:{}|abnornalities entried:{}'.format(len(nlst_ctab_idc['pid'].unique()),len(nlst_ctab_idc)))

MARGIN_DECODER_Classification       = {"Spiculated (Stellate)":'spiculation',"Smooth":'smooth',"Poorly defined":"poorly-defined","Unable to determine":"poorly-defined"}
ATTN_CLASSIFICATION_MAP             = {"Soft Tissue": "solid","Ground glass": "ground-glass","Mixed": "part-solid","Fluid/water": "part-solid","Fat": "solid","Other": "others","Unable to determine": "others"}

nlst_pancandata                   = nlst_ctab_idc

# 构建新列“家族史”
nlst_pancandata['Family_History'] = nlst_pancandata[['fambrother', 'famchild', 'famfather', 'fammother', 'famsister']].max(axis=1)


##--- Making an clean dataframe from feature importance understanding

columns_for_ml_analysis = ['pid','study_yr','age','gender','all_sct_set','sct_long_dia','sct_margins','sct_pre_att','sct_epi_loc','can_scr','diagemph','Family_History']
                           
nlst_ct_nodule_df       = nlst_pancandata[columns_for_ml_analysis]

#--- Cleaning --

# 去掉NA
nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['sct_long_dia']).reset_index(drop=True)

print('3) Drop missing nodule sizes |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))

nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['sct_pre_att']).reset_index(drop=True)

print('4) Drop missing nodule type  |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))


nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['all_sct_set']).reset_index(drop=True)

print('5) Drop missing Split Set    |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))

nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['Family_History']).reset_index(drop=True)

print('6) Drop missing Family History    |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))


nlst_ct_nodule_df = nlst_ct_nodule_df.dropna(subset=['diagemph']).reset_index(drop=True)

print('7) Drop missing Emphysema info    |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))





sct_pre_att_mapping    = {"Soft Tissue": "solid","Ground glass": "ground-glass","Mixed": "part-solid","Fluid/water": "part-solid","Fat": "solid","Other": "others","Unable to determine": "others"}




# 构建新列nodule type，drop掉type为others的行
nlst_ct_nodule_df['Nodule_Type'] = nlst_ct_nodule_df['sct_pre_att'].map(sct_pre_att_mapping)
nlst_ct_nodule_df                = nlst_ct_nodule_df[nlst_ct_nodule_df['Nodule_Type']!='others'].reset_index(drop=True)
print('8) Filtered by Solid, part-solid, ground-glass |patient:{}|abnornalities entried:{}'.format(len(nlst_ct_nodule_df['pid'].unique()),len(nlst_ct_nodule_df)))




# 构建新列Spiculation
sct_margins_mapping               = {"Spiculated (Stellate)":'spiculation',"Smooth":'smooth',"Poorly defined":"poorly-defined","Unable to determine":"poorly-defined"}
nlst_ct_nodule_df['Spiculation']  = nlst_ct_nodule_df['sct_margins'].map(sct_margins_mapping)
nlst_ct_nodule_df['Spiculation']  = nlst_ct_nodule_df['Spiculation'].apply(lambda x: 1 if 'spiculation' in x else 0)



# 根据位置sct_epi_loc'，构建新列'Upper_Lobe'，如果位置在upper或middle lobe，返回1，否则返回0
sct_epi_loc_mapping = {"Right Upper Lobe":'Upper',
                       "Right Middle Lobe":'Upper',
                       "Right Lower Lobe":'Lower',
                       "Left Upper Lobe":'Upper',
                       "Lingula": 'Other',
                       "Left Lower Lobe":'Lower',
                       "Other":'Other'}

nlst_ct_nodule_df['Upper_Lobe']     = nlst_ct_nodule_df['sct_epi_loc'].map(sct_epi_loc_mapping)
nlst_ct_nodule_df['Upper_Lobe']     = nlst_ct_nodule_df['Upper_Lobe'].apply(lambda x: 1 if 'Upper' in x or 'Middle' in x else 0)


C:\Users\YiJin\AppData\Local\Temp\ipykernel_34904\105657882.py:159: DtypeWarning: Columns (239,240,348) have mixed types. Specify dtype option on import or set low_memory=False.
  all_participants                             = pd.read_csv(DF_SCR_PATH+'participant_d040722.csv')


1) all data |patient:24517|abnornalities entried:177651
2) Filtered by non-calcified nodule > 4mm |patient:19036|abnornalities entried:80553
3) Drop missing nodule sizes |patient:10239|abnornalities entried:35239
4) Drop missing nodule type  |patient:10238|abnornalities entried:35238
5) Drop missing Split Set    |patient:10204|abnornalities entried:35184
6) Drop missing Family History    |patient:10057|abnornalities entried:34659
7) Drop missing Emphysema info    |patient:10040|abnornalities entried:34571
8) Filtered by Solid, part-solid, ground-glass |patient:9795|abnornalities entried:33031


## construct a new y

目前想法：先根据'scr_days0','scr_days1','scr_days2','candx_days'构建一个新dataframe，存储每个patient在第几次screening发现了癌症。之后用ct data构建一个新列，显示这一行是属于第几个screening，最后合并。

In [4]:
ct_ab  = pd.read_csv(DF_SCR_PATH+"Spiral CT Abnormalities/sct_abnormalities_d040722.csv")

In [ ]:
test = ct_ab[ct_ab['sct_ab_desc'].isin(ABNORNALITIES_TO_CONDIDER)]

In [3]:
# 构建新的y
test = all_participants[['pid','scr_days0','scr_days1','scr_days2','candx_days']]

## 保存

In [ ]:
#---- Set1 and Set2 Division=====#

nlst_ct_nodule_df_set1 = nlst_ct_nodule_df[nlst_ct_nodule_df['all_sct_set']==1].reset_index(drop=True)
nlst_ct_nodule_df_set2 = nlst_ct_nodule_df[nlst_ct_nodule_df['all_sct_set']==2].reset_index(drop=True)


print('Devide by CT Sets')
print('9a) Set-1 |patient:{}|abnornalities entried:{}|Cancer:{}'.format(len(nlst_ct_nodule_df_set1['pid'].unique()),len(nlst_ct_nodule_df_set1),len(nlst_ct_nodule_df_set1[nlst_ct_nodule_df_set1['Cancer_lbl']==1])))
print('9b) Set-2 |patient:{}|abnornalities entried:{}|Cancer:{}'.format(len(nlst_ct_nodule_df_set2['pid'].unique()),len(nlst_ct_nodule_df_set2),len(nlst_ct_nodule_df_set2[nlst_ct_nodule_df_set2['Cancer_lbl']==1])))


save_dir = 'ml_dataset/'

nlst_ct_nodule_df_set1.to_csv(save_dir +'nlst_ct_nodule_df_set1.csv',index=False,encoding='utf-8')
nlst_ct_nodule_df_set2.to_csv(save_dir +'nlst_ct_nodule_df_set2.csv',index=False,encoding='utf-8')